# Design Patterns in Python

This notebook provides concise, runnable examples for: 

- **Behavioral**: Chain of Responsibility, State, Command, Observer, Strategy
- **Structural**: Adapter, Bridge, Composite, Decorator, Facade
- **Creational**: Abstract Factory, Factory Method, Builder, Singleton

## Behavioral Patterns

In [ ]:
# Chain of Responsibility
from __future__ import annotations
from abc import ABC, abstractmethod

class Handler(ABC):
    def __init__(self):
        self._next = None

    def set_next(self, handler: 'Handler') -> 'Handler':
        self._next = handler
        return handler

    @abstractmethod
    def handle(self, request: int):
        if self._next:
            return self._next.handle(request)
        return f'No handler for {request}'

class NegativeHandler(Handler):
    def handle(self, request: int):
        if request < 0:
            return f'NegativeHandler handled {request}'
        return super().handle(request)

class ZeroHandler(Handler):
    def handle(self, request: int):
        if request == 0:
            return f'ZeroHandler handled {request}'
        return super().handle(request)

class PositiveHandler(Handler):
    def handle(self, request: int):
        if request > 0:
            return f'PositiveHandler handled {request}'
        return super().handle(request)

chain = NegativeHandler()
chain.set_next(ZeroHandler()).set_next(PositiveHandler())

for value in [-3, 0, 7]:
    print(chain.handle(value))

In [ ]:
# State
from abc import ABC, abstractmethod

class State(ABC):
    @abstractmethod
    def handle(self, player: 'MusicPlayer') -> None:
        pass

class PlayingState(State):
    def handle(self, player: 'MusicPlayer') -> None:
        print('Pausing music')
        player.state = PausedState()

class PausedState(State):
    def handle(self, player: 'MusicPlayer') -> None:
        print('Resuming music')
        player.state = PlayingState()

class MusicPlayer:
    def __init__(self):
        self.state: State = PausedState()

    def press_play_pause(self):
        self.state.handle(self)

player = MusicPlayer()
player.press_play_pause()
player.press_play_pause()
player.press_play_pause()

In [ ]:
# Command
from abc import ABC, abstractmethod

class Command(ABC):
    @abstractmethod
    def execute(self) -> None:
        pass

class Light:
    def on(self):
        print('Light is ON')

    def off(self):
        print('Light is OFF')

class LightOnCommand(Command):
    def __init__(self, light: Light):
        self.light = light

    def execute(self) -> None:
        self.light.on()

class LightOffCommand(Command):
    def __init__(self, light: Light):
        self.light = light

    def execute(self) -> None:
        self.light.off()

class RemoteControl:
    def submit(self, command: Command):
        command.execute()

light = Light()
remote = RemoteControl()
remote.submit(LightOnCommand(light))
remote.submit(LightOffCommand(light))

In [ ]:
# Observer
from abc import ABC, abstractmethod

class Observer(ABC):
    @abstractmethod
    def update(self, temperature: float) -> None:
        pass

class Subject:
    def __init__(self):
        self._observers = []

    def attach(self, observer: Observer):
        self._observers.append(observer)

    def detach(self, observer: Observer):
        self._observers.remove(observer)

    def notify(self, temperature: float):
        for observer in self._observers:
            observer.update(temperature)

class PhoneDisplay(Observer):
    def update(self, temperature: float) -> None:
        print(f'Phone: temperature is {temperature}C')

class WindowDisplay(Observer):
    def update(self, temperature: float) -> None:
        print(f'Window: temperature is {temperature}C')

weather_station = Subject()
phone = PhoneDisplay()
window = WindowDisplay()
weather_station.attach(phone)
weather_station.attach(window)
weather_station.notify(22.5)

In [ ]:
# Strategy
from abc import ABC, abstractmethod

class SortStrategy(ABC):
    @abstractmethod
    def sort(self, data):
        pass

class AscendingSort(SortStrategy):
    def sort(self, data):
        return sorted(data)

class DescendingSort(SortStrategy):
    def sort(self, data):
        return sorted(data, reverse=True)

class Context:
    def __init__(self, strategy: SortStrategy):
        self.strategy = strategy

    def set_strategy(self, strategy: SortStrategy):
        self.strategy = strategy

    def execute(self, data):
        return self.strategy.sort(data)

nums = [5, 1, 4, 2]
context = Context(AscendingSort())
print('Ascending:', context.execute(nums))
context.set_strategy(DescendingSort())
print('Descending:', context.execute(nums))

## Structural Patterns

In [ ]:
# Adapter
class OldPrinter:
    def print_text(self, text: str):
        return f'OldPrinter: {text}'

class NewPrinterInterface:
    def print(self, text: str):
        raise NotImplementedError

class PrinterAdapter(NewPrinterInterface):
    def __init__(self, old_printer: OldPrinter):
        self.old_printer = old_printer

    def print(self, text: str):
        return self.old_printer.print_text(text)

printer = PrinterAdapter(OldPrinter())
print(printer.print('Hello through adapter'))

In [ ]:
# Bridge
from abc import ABC, abstractmethod

class Device(ABC):
    @abstractmethod
    def on(self):
        pass

    @abstractmethod
    def off(self):
        pass

class TV(Device):
    def on(self):
        print('TV ON')

    def off(self):
        print('TV OFF')

class Radio(Device):
    def on(self):
        print('Radio ON')

    def off(self):
        print('Radio OFF')

class Remote:
    def __init__(self, device: Device):
        self.device = device

    def power_on(self):
        self.device.on()

    def power_off(self):
        self.device.off()

tv_remote = Remote(TV())
radio_remote = Remote(Radio())
tv_remote.power_on()
radio_remote.power_on()

In [ ]:
# Composite
from abc import ABC, abstractmethod

class FileSystemComponent(ABC):
    @abstractmethod
    def show(self, depth=0):
        pass

class File(FileSystemComponent):
    def __init__(self, name):
        self.name = name

    def show(self, depth=0):
        print('  ' * depth + f'- File: {self.name}')

class Folder(FileSystemComponent):
    def __init__(self, name):
        self.name = name
        self.children = []

    def add(self, component: FileSystemComponent):
        self.children.append(component)

    def show(self, depth=0):
        print('  ' * depth + f'+ Folder: {self.name}')
        for child in self.children:
            child.show(depth + 1)

root = Folder('root')
root.add(File('README.md'))
src = Folder('src')
src.add(File('main.py'))
src.add(File('utils.py'))
root.add(src)
root.show()

In [ ]:
# Decorator
from abc import ABC, abstractmethod

class Coffee(ABC):
    @abstractmethod
    def cost(self):
        pass

    @abstractmethod
    def description(self):
        pass

class SimpleCoffee(Coffee):
    def cost(self):
        return 3

    def description(self):
        return 'Simple Coffee'

class CoffeeDecorator(Coffee):
    def __init__(self, coffee: Coffee):
        self.coffee = coffee

class MilkDecorator(CoffeeDecorator):
    def cost(self):
        return self.coffee.cost() + 1

    def description(self):
        return self.coffee.description() + ', Milk'

class SugarDecorator(CoffeeDecorator):
    def cost(self):
        return self.coffee.cost() + 0.5

    def description(self):
        return self.coffee.description() + ', Sugar'

coffee = SugarDecorator(MilkDecorator(SimpleCoffee()))
print(coffee.description())
print('Cost:', coffee.cost())

In [ ]:
# Facade
class CPU:
    def freeze(self):
        print('CPU freeze')

    def execute(self):
        print('CPU execute')

class Memory:
    def load(self):
        print('Memory load')

class HardDrive:
    def read(self):
        print('HardDrive read')

class ComputerFacade:
    def __init__(self):
        self.cpu = CPU()
        self.memory = Memory()
        self.hard_drive = HardDrive()

    def start(self):
        self.cpu.freeze()
        self.memory.load()
        self.hard_drive.read()
        self.cpu.execute()

computer = ComputerFacade()
computer.start()

## Creational Patterns

In [ ]:
# Abstract Factory
from abc import ABC, abstractmethod

class Button(ABC):
    @abstractmethod
    def paint(self):
        pass

class Checkbox(ABC):
    @abstractmethod
    def paint(self):
        pass

class WinButton(Button):
    def paint(self):
        print('Render Windows button')

class MacButton(Button):
    def paint(self):
        print('Render Mac button')

class WinCheckbox(Checkbox):
    def paint(self):
        print('Render Windows checkbox')

class MacCheckbox(Checkbox):
    def paint(self):
        print('Render Mac checkbox')

class GUIFactory(ABC):
    @abstractmethod
    def create_button(self) -> Button:
        pass

    @abstractmethod
    def create_checkbox(self) -> Checkbox:
        pass

class WinFactory(GUIFactory):
    def create_button(self) -> Button:
        return WinButton()

    def create_checkbox(self) -> Checkbox:
        return WinCheckbox()

class MacFactory(GUIFactory):
    def create_button(self) -> Button:
        return MacButton()

    def create_checkbox(self) -> Checkbox:
        return MacCheckbox()

def render_ui(factory: GUIFactory):
    button = factory.create_button()
    checkbox = factory.create_checkbox()
    button.paint()
    checkbox.paint()

render_ui(WinFactory())
render_ui(MacFactory())

In [ ]:
# Factory Method
from abc import ABC, abstractmethod

class Transport(ABC):
    @abstractmethod
    def deliver(self):
        pass

class Truck(Transport):
    def deliver(self):
        print('Deliver by land in a truck')

class Ship(Transport):
    def deliver(self):
        print('Deliver by sea in a ship')

class Logistics(ABC):
    @abstractmethod
    def create_transport(self) -> Transport:
        pass

    def plan_delivery(self):
        transport = self.create_transport()
        transport.deliver()

class RoadLogistics(Logistics):
    def create_transport(self) -> Transport:
        return Truck()

class SeaLogistics(Logistics):
    def create_transport(self) -> Transport:
        return Ship()

RoadLogistics().plan_delivery()
SeaLogistics().plan_delivery()

In [ ]:
# Builder
class House:
    def __init__(self):
        self.parts = []

    def add(self, part: str):
        self.parts.append(part)

    def __str__(self):
        return 'House with: ' + ', '.join(self.parts)

class HouseBuilder:
    def __init__(self):
        self.house = House()

    def build_walls(self):
        self.house.add('walls')
        return self

    def build_roof(self):
        self.house.add('roof')
        return self

    def build_garage(self):
        self.house.add('garage')
        return self

    def result(self) -> House:
        return self.house

house = HouseBuilder().build_walls().build_roof().build_garage().result()
print(house)

In [ ]:
# Singleton
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Logger(metaclass=SingletonMeta):
    def __init__(self):
        self.messages = []

    def log(self, message: str):
        self.messages.append(message)

a = Logger()
b = Logger()
a.log('first entry')
print('Same instance:', a is b)
print('Messages from b:', b.messages)